# Optimization of the NACA parameters

This is a test to see how optuna performs to find the best NACA parameters to optimize the aerodynamic finess *(Lift/Drag)*

In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath(".."))

from src.geometry import AirfoilGeometry
from src.physics import AeroSolver
import optuna

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


For this optimization velocity, altitude, chord and alpha are fixed

In [3]:
V_FIXED = 100.0
ALT_FIXED = 100     
CHORD_FIXED = 0.5 
ALPHA_TARGET = 5

Only the m, p and t NACA parameters are optimized by Optuna

In [4]:
def objective_naca(trial):
    m = trial.suggest_int("m", 0, 9)
    p = trial.suggest_int("p", 0, 9)
    t = trial.suggest_int("t", 1, 40)
    
    af = AirfoilGeometry(n_points=100)
    xu, yu, xl, yl = af.generate_naca4(m_int=m, p_int=p, t_int=t)
    x_coords, y_coords = af.get_coords_for_solver(xu, yu, xl, yl)
    
    solver = AeroSolver(altitude=ALT_FIXED, velocity=V_FIXED, chord=CHORD_FIXED)
    res = solver.solve(x_coords, y_coords, alpha=ALPHA_TARGET)
    
    cl = res['cl'][0]
    cd = res['cd'][0]
    
    if cd <= 0: return 0
    
    return cl / cd

study = optuna.create_study(direction="maximize")
study.optimize(objective_naca, n_trials=50)

best_naca = f"{study.best_params['m']}{study.best_params['p']}{study.best_params['t']:02d}"
print(f"\nBest : NACA {best_naca}")
print(f"Max Finess (Lift/Drag) : {study.best_value:.2f}")
optuna.visualization.plot_optimization_history(study)

[I 2026-03-27 12:04:47,427] A new study created in memory with name: no-name-fb90a7c3-9427-43d0-8297-d72cbf1d8197
[I 2026-03-27 12:04:47,497] Trial 0 finished with value: 97.60555833887913 and parameters: {'m': 1, 'p': 8, 't': 6}. Best is trial 0 with value: 97.60555833887913.
[I 2026-03-27 12:04:47,497] Trial 1 finished with value: 105.68474985893064 and parameters: {'m': 6, 'p': 7, 't': 10}. Best is trial 1 with value: 105.68474985893064.
[I 2026-03-27 12:04:47,497] Trial 2 finished with value: 69.07202385331833 and parameters: {'m': 3, 'p': 7, 't': 30}. Best is trial 1 with value: 105.68474985893064.
[I 2026-03-27 12:04:47,497] Trial 3 finished with value: 46.035769128675746 and parameters: {'m': 3, 'p': 3, 't': 36}. Best is trial 1 with value: 105.68474985893064.
[I 2026-03-27 12:04:47,512] Trial 4 finished with value: 14.22337065406673 and parameters: {'m': 0, 'p': 1, 't': 2}. Best is trial 1 with value: 105.68474985893064.
[I 2026-03-27 12:04:47,517] Trial 5 finished with value: 


Best : NACA 7415
Max Finess (Lift/Drag) : 193.12


In [5]:
optuna.visualization.plot_contour(study, params=['m', 'p', 't'])


In [6]:
optuna.visualization.plot_param_importances(study)